# Exploring the TeNPy library

This notebook uses the TeNPy libary - if you didn't do that yet, now you need to install tenpy.
Run
```bash
conda install -c conda-forge physics-tenpy
```
or 
```bash
pip install physics-tenpy
```

When running this notebook on https://colab.research.google.com, uncomment and run the cells below (removing the `#`).

Alternatively, you can run this notebook locally with jupyter, provided that you have the `toycodes` subfolder from 
https://github.com/tenpy/tenpy_toycodes
in the same folder as your notebook.

In [ ]:
#!pip install git+https://github.com/tenpy/tenpy_toycodes.git
#!pip install git+https://github.com/tenpy/tenpy.git

# use `pip uninstall tenpy-toycodes physics-tenpy` to remove them again.

You can add your code below by inserting additional cells as neccessary and running them (press Shift+Enter).

**DISCLAIMER**: Like for the toy codes, we only use very small bond dimensions here. For state-of-the-art MPS calculations (especially for cylinders towards 2D), `chi` should be significantly larger, often on the order of several 1000s.

In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
from pprint import pprint

np.set_printoptions(precision=5, suppress=True, linewidth=100)
plt.rcParams['figure.dpi'] = 150

In [ ]:
import tenpy
import tenpy.linalg.np_conserved as npc
from tenpy.algorithms import tebd, dmrg, tdvp
from tenpy.networks.site import SpinHalfSite, SpinSite, FermionSite
from tenpy.networks.mps import MPS
from tenpy.models.tf_ising import TFIChain

tenpy.tools.misc.setup_logging(to_stdout="INFO")

## Overview

The source code of TeNPy is at https://github.com/tenpy/tenpy/; 
you can find links to the documentation and the forum in the Readme there.

The [**documentation**](https://tenpy.readthedocs.io) is roughly split into the "User guide" (upper part in the left side-bar) and the reference of all the functions and classes (lower part).

In the documentation, you can also find an  [overview](https://tenpy.readthedocs.io/en/latest/intro/overview.html) with basic examples to get started.

Here, we will rather look at the examples from the slides.

### Exercise: Example contractions

[TeNPy Arrays](https://tenpy.readthedocs.io/en/latest/reference/tenpy.linalg.np_conserved.Array.html) are like numpy arrays, in that you can do basic tensor operations like `npc.tensordot` to contract them and matrix decompositions like `npc.eigh`, `npc.qr` and `npc.svd` from the [np_conserved](https://tenpy.readthedocs.io/en/latest/reference/tenpy.linalg.np_conserved.html) module with them.
However, TeNPy arrays have additional structure of "charges" on the legs to exploit a block-sparse structure, thus you can no longer do a simple `reshape` - but need to use [combine_legs](https://tenpy.readthedocs.io/en/latest/reference/tenpy.linalg.np_conserved.Array.html#tenpy.linalg.np_conserved.Array.combine_legs) to group legs, and you can only [split_legs](https://tenpy.readthedocs.io/en/latest/reference/tenpy.linalg.np_conserved.Array.html#tenpy.linalg.np_conserved.Array.split_legs) on legs that have been combined before.

The first example on the slides uses `the SpinHalfSite` to initize `npc.Array` with charge conservation.

Try to run the examle.
Print some of the tensors, and check whether they are what you expect.
Note that just printing them shows the labels the tensors have, with a very similar convetion to our toy codes!
You can also use `Z1.to_ndarray()` to convert TeNPy Arrays to basic numpy arrays - but for the reverse direction, you need to specify the LegCharge, so this is more difficult. In practice, you get more and more conve

In [ ]:
import tenpy.linalg.np_conserved as npc
from tenpy.networks.site import SpinHalfSite

s = SpinHalfSite(conserve='Sz')
Z1 = s.get_op('Sigmaz')
Z1 = Z1.replace_labels(['p', 'p*'],
                          ['j1', 'j1*'])
Z2 = s.get_op('Sigmaz')
Z2 = Z2.replace_labels(['p', 'p*'],
                          ['j2', 'j2*'])

h = npc.outer(Z1, Z2)
h_m = h.combine_legs([['j1', 'j2'],
                      ['j1*', 'j2*']])

E, V = npc.eigh(h_m)
U = npc.expm(-1.j * 0.1 * h_m)

### Exercise: First MPS

The MPS example (for smaller `L=8` to avoid excessive output) from the slides is given below. Run it!

Modify it to initialize a Neel state instead.

Use the documentation of the [MPS class](https://tenpy.readthedocs.io/en/latest/reference/tenpy.networks.mps.MPS.html) to find out how to apply a local operator.
Create another copy of the Neel state by initializing an Up-state MPS and applying the $S^-$ operator with the name `"Sm"` on every other site. Check that the `overlap` with the directly initialized Neel state is 1.


In [ ]:
from tenpy import SpinHalfSite, MPS

s = SpinHalfSite(conserve='Sz')

L = 8
product_state = ["up"] * L
psi_Neel = MPS.from_product_state([s] * L, product_state, bc='finite')
print("entropy = ", psi_Neel.entanglement_entropy())
print("<Z_i> = ", psi_Neel.expectation_value("Sigmaz"))
print("<Z_i Z_j> = ", psi_Neel.correlation_function("Sigmaz", "Sigmaz"))


## Initializing a Model

In TeNPy, the model defines the Hilbert space and local operators, and ultimately fixes whether charge conservation is used. Therefore, you should usually start with the initialization of the model.
There are many predefined models in `tenpy.models`, that you can often just use.

We will look at the transverse field Ising model as an example. One advantage of TeNPy is that it can exploit (abelian) charge conservation for speedups, e.g. the transverse field Ising model preserves an overall spin parity. However, this requires the form 
$$ H = - J \sum_{i} \sigma^x_i \sigma^x_{i+1} - g \sum_{i} \sigma^z_i  \textrm{ in TeNPy}$$
compared to the form 
$$ H = - J \sum_{i} \sigma^z_i \sigma^z_{i+1} - g \sum_{i} \sigma^x_i \textrm{  (not suitable for charge conservation)}$$ you might be more familiar with, where X and Z are exchanged.

But if we dind't know that, our first attemt might rather look like this:

In [ ]:
from tenpy import SpinHalfSite, CouplingMPOModel

class TFIModel(CouplingMPOModel):
    def init_sites(self, model_params):
        return SpinHalfSite(conserve=None)

    def init_terms(self, model_params):
        J = model_params.get('J', 1.)
        g = model_params.get('g', 1.)
        for u in range(len(self.lat.unit_cell)):
            self.add_onsite(-g, u, 'Sigmax')
        for u1, u2, dx in self.lat.pairs['nearest_neighbors']:
            self.add_coupling(-J, u1, 'Sigmaz', u2, 'Sigmaz', dx)

model = TFIModel(dict(g=1.1, L=8))


### Exercise: verify the model

Implementing your own model is probably the first task you face when you want to use TeNPy for your own project. For that, look at the [Model Intro of the documentation](https://tenpy.readthedocs.io/en/latest/intro/model.html).
There is a section there how to "verify" that a model actually implements what you expected.
Use that to print all the terms in the above initialized `model`.

Switching X and Z and enabling charge conservation, we arrive on the slides at the following cell.
Verify that the terms switched accordingly!

In [ ]:
from tenpy import SpinHalfSite, CouplingMPOModel

class TFIModel(CouplingMPOModel):
    def init_sites(self, model_params):
        return SpinHalfSite(conserve='parity')

    def init_terms(self, model_params):
        J = model_params.get('J', 1.)
        g = model_params.get('g', 1.)
        for u in range(len(self.lat.unit_cell)):
            self.add_onsite(-g, u, 'Sigmaz')
        for u1, u2, dx in self.lat.pairs['nearest_neighbors']:
            self.add_coupling(-J, u1, 'Sigmax', u2, 'Sigmax', dx)

model = TFIModel(dict(g=1.1, L=8))


Note that TeNPy writes the default parameters used back into the dictionary, which can be a quick way to explore what options you can set, as shown in the following cell.
Alternatively, find the "options" section in the documentation of the corresponding class, 
e.g. [here for the TFIModel](https://tenpy.readthedocs.io/en/latest/reference/tenpy.models.tf_ising.TFIModel.html#cfg-config-TFIModel).

In [ ]:
from tenpy.models.tf_ising import TFIModel
from pprint import pprint

model_params = {
    'L': 20,
    'g': 1.0,
    'bc_MPS': 'finite',
    'conserve': 'best'
}
model = TFIModel(model_params)
# you can now print the default parameters used:
print("used parameters, including default/not specified ones:")
pprint(model_params)

That tells us that switching to a 2D model can then be as easy as replacing the `lattice` parameter.
For more details read the corresponding user guides:
- [Models (Introduction)](https://tenpy.readthedocs.io/en/latest/intro/model.html)
- [Details on the implementation of Models](https://tenpy.readthedocs.io/en/latest/intro/model_details.html)
- [Details on the lattice geometry](https://tenpy.readthedocs.io/en/latest/intro/lattices.html)

In [ ]:
model_2D = TFIModel(dict(g=1.1, lattice='Square', Lx=5, Ly=4))

Since the model defines what charges we can conserver, we typically *first* intialize the model, and then the MPS given the model. This can be done like this:

In [ ]:
from tenpy.models.tf_ising import TFIChain

model = TFIChain(dict(g=0.1, J=1., L=8))
p_state = ['up'] * model.lat.N_sites
psi = MPS.from_product_state(model.lat.mps_sites(), p_state, bc=model.lat.bc_MPS)

### Exercise: measure bond energies

Check the [Model.bond_energies](https://tenpy.readthedocs.io/en/latest/reference/tenpy.models.model.NearestNeighborModel.html#tenpy.models.model.NearestNeighborModel.bond_energies) for the all-up product state. Why do you have differnt values on the boundaries?.

## Running DMRG

Given the model and state, running DMRG isn't hard.
Again, there are many (default) parameters for fine-tuning, see [this full option list](https://tenpy.readthedocs.io/en/latest/reference/tenpy.algorithms.dmrg.TwoSiteDMRGEngine.html#cfg-config-TwoSiteDMRGEngine) for details.

In [ ]:
from tenpy import TFIModel, MPS, TwoSiteDMRGEngine

model_params = dict(L=30, g=1.1, conserve=None)
M = TFIModel(model_params)
product_state = ["up"] * M.lat.N_sites
psi = MPS.from_product_state(M.lat.mps_sites(),
                                product_state,
                                bc=M.lat.bc_MPS)
algorithm_params = dict(trunc_params=dict(chi_max=50))
alg = TwoSiteDMRGEngine(psi, M, algorithm_params)
alg.run()   # can run a while...


### Exercise: Expectation values after DMRG run

Print the entroy, expectation values of Z_i for the final state `psi` after the DMRG run.

### Exercise: infinite MPS

Read the documentation to find out how to run DMRG for `'infinite'` MPS. 
(You need to initialize a new model, state, and DMRG engine for this.)

### Advanced Exercise: DMRG phase diagram for different `g`

Use infinite DMRG to get the ground state phase diagram of the TF Ising model.
To do that, run DMRG for different `g`, each time measuring entriopy, X, Z expectation values and the XX correlations between spins 0 and 20.
Plot it.

# Simulations

For large-scale simulations, especially when running on a computing cluster, we recommend looking into the tenpy Simulation setup.

For that, read the [user guide on Simulations](https://tenpy.readthedocs.io/en/latest/intro/simulations.html).

You can explore the setup in a jupyter notebook as illustrated in the following cell.
Note that you should install the packages `pyyaml` and `h5py` for this:
```bash
pip install pyyaml h5py
# or
conda install pyyaml h5py
```




In [ ]:
import yaml

In [ ]:
simulation_params = tenpy.load_yaml_with_py_eval(yaml_content="""
simulation_class : GroundStateSearch
output_filename: DMRG_sim_example.pkl

model_class:  TFIModel
model_params:
    L : 32
    bc_MPS : finite

initial_state_params:
    method : lat_product_state
    product_state : [[up]]

algorithm_class: TwoSiteDMRGEngine
algorithm_params:
    trunc_params:
        chi_max: 100
""")
pprint(simulation_params)

In [ ]:
tenpy.run_simulation(**simulation_params)